In [ ]:
import subprocess
import sys
import shutil

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    requirements = [
        'langchain==1.3.13',
        'langchain-core==1.4.9',
        'langchain-openai==1.3.5',
        'langsmith==0.9.5',
        'python-dotenv==1.0.1',
        'pyyaml==6.0.2',
    ]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *requirements])
    print('Node:', shutil.which('node'), '| npx:', shutil.which('npx'))
else:
    print('Entorno local detectado. Usa el entorno uv (.venv) descrito en README para manejar dependencias.')
    if shutil.which('npx') is None:
        print('ADVERTENCIA: no se encontro `npx`. Instala Node.js >= 20 (https://nodejs.org) para usar la CLI de Packmind.')

# Versionamiento de skills, standards y commands con Packmind

En la **Leccion 2** versionamos *prompts* con el Prompt Hub de LangSmith. Aqui damos el
siguiente paso y versionamos los tres artefactos que **[Packmind](https://app.packmind.ai)**
gestiona para un agente de codigo:

| Artefacto | Como se usa | Donde vive (agente Claude) |
|---|---|---|
| **Skill** | *agent-discovered*: el agente la carga **bajo demanda** | `.claude/skills/<slug>/SKILL.md` |
| **Standard** | *siempre activo*: regla que esta en el contexto todo el tiempo | `.claude/rules/<slug>.md` |
| **Command** | *invocado por el usuario* (estilo slash-command) | `.claude/commands/<slug>.md` |

Los tres se **crean como archivos**, se **versionan** con `playbook add`/`submit`, se **agrupan**
en un *package* y se **despliegan** juntos. Luego construimos un **agente LangChain** que los
consume: el standard entra siempre al system prompt y la skill se carga bajo demanda
(*progressive disclosure*). Eso es *context engineering*: en el contexto, solo lo relevante,
solo cuando hace falta.

## Importaciones y variables
Usaremos la CLI de Packmind (via `npx`) para el ciclo de vida de los artefactos, y LangChain +
OpenAI + LangSmith para construir y trazar el agente que los consume.

In [ ]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
os.environ.setdefault('LANGSMITH_PROJECT', os.getenv('LANGSMITH_PROJECT', 'clase-3-6-l3-skills'))
os.environ.setdefault('LANGSMITH_TRACING', 'true')

REQUIRED_KEYS = ('OPENAI_API_KEY', 'LANGSMITH_API_KEY', 'PACKMIND_API_KEY_V3')

if IN_COLAB:
    from google.colab import userdata

    for key in REQUIRED_KEYS:
        secret_value = userdata.get(key)
        if not secret_value:
            raise ValueError(f'Falta la variable {key} en el gestor de secretos de Colab')
        os.environ[key] = secret_value
else:
    try:
        from dotenv import load_dotenv
    except ImportError as exc:  # pragma: no cover
        raise ImportError('Instala python-dotenv para cargar las variables locales.') from exc
    env_file = Path('.env')
    if env_file.exists():
        load_dotenv(env_file)
        print('Variables cargadas desde .env')
    else:
        print('No se encontro .env en el root. Se usaran las variables de entorno del sistema.')

missing_keys = [key for key in REQUIRED_KEYS if not os.getenv(key)]
if missing_keys:
    raise EnvironmentError(f'Faltan variables requeridas: {", ".join(missing_keys)}')

## Un ayudante para la CLI de Packmind

La CLI se ejecuta con `npx @packmind/cli ...`. La envolvemos en una funcion `packmind(...)` que:
- captura la salida para poder **inspeccionarla/parsearla**,
- hereda las variables de entorno (asi la CLI ve `PACKMIND_API_KEY_V3` y se autentica sola),
- por defecto corre **sin stdin**; para comandos interactivos (como `init`) le pasamos la
  respuesta con `stdin_text=...` para no colgar el kernel.

In [ ]:
BASE_DIR = Path.cwd()
SKILL_REPO = BASE_DIR / 'skill_repo'       # proyecto donde AUTORAMOS y versionamos los artefactos
WORKDIR = BASE_DIR / 'agent_workdir'       # proyecto CONSUMIDOR: instalamos el package aqui

SKILL_SLUG = 'reportero-financiero-cl'     # skill  -> .claude/skills/<slug>/SKILL.md
STANDARD_SLUG = 'formato-montos-clp'       # standard -> .claude/rules/<slug>.md
COMMAND_SLUG = 'reporte-trimestral'        # command  -> .claude/commands/<slug>.md


import time


def packmind(*args, cwd=BASE_DIR, check=True, stdin_text=None, retries=0):
    """Ejecuta `npx @packmind/cli <args>` y devuelve el CompletedProcess.

    - Por defecto corre sin stdin (no interactivo). Para comandos interactivos como `init`,
      pasa `stdin_text=...` con la respuesta que enviarias por teclado.
    - `retries`: los comandos que hablan con el servidor de Packmind a veces devuelven un
      500 transitorio ("Internal server error"). Como nuestras operaciones son idempotentes,
      reintentamos hasta `retries` veces antes de rendirnos.
    """
    kwargs = dict(cwd=str(cwd), capture_output=True, text=True, env=os.environ.copy())
    if stdin_text is not None:
        kwargs['input'] = stdin_text
    else:
        kwargs['stdin'] = subprocess.DEVNULL
    attempt = 0
    while True:
        result = subprocess.run(['npx', '--yes', '@packmind/cli', *args], **kwargs)
        if result.returncode == 0 or attempt >= retries:
            print((result.stdout or '') + (result.stderr or ''))
            if check and result.returncode != 0:
                raise RuntimeError(f"packmind-cli {' '.join(args)} devolvio codigo {result.returncode}")
            return result
        attempt += 1
        print(f'[reintento {attempt}/{retries}] `packmind {args[0]}` fallo (codigo {result.returncode}); reintentando en 3s...')
        time.sleep(3)

### 1. Verifica tu sesion
`whoami` confirma que la API key (`PACKMIND_API_KEY_V3`) es valida y muestra tu organizacion.

In [ ]:
packmind('whoami')

### 2. Inicializa el repo de artefactos

`init` crea `packmind.json`, registra el agente destino y genera la carpeta `.claude/`
(con las skills por defecto de Packmind). **Es interactivo**: pide elegir los agentes con un
menu numerado. Le enviamos `2` (Claude Code) por stdin para que no se cuelgue el kernel.

> El notebook **parte de cero** cada corrida (borra `skill_repo/` antes de `init`). Asi evitamos
> estados locales inconsistentes de corridas previas: por ejemplo, un `packmind-lock.json` que
> apunta a un artefacto ya borrado en el servidor hace que `playbook add` falle con
> *"Failed to check artifact version"*.

In [ ]:
import shutil as _sh

# Partimos de cero para evitar estados locales inconsistentes de corridas previas.
if SKILL_REPO.exists():
    _sh.rmtree(SKILL_REPO)
SKILL_REPO.mkdir(parents=True, exist_ok=True)

# init pregunta por los agentes destino; "2" = Claude Code -> los artefactos viven en .claude/
packmind('init', cwd=SKILL_REPO, stdin_text='2\n', check=False, retries=2)

### 3. Crea la skill (v1)

En Packmind cada artefacto vive en un **directorio reconocido**. Para una **skill** (agente
Claude) es `.claude/skills/<slug>/SKILL.md`, con:
- **Frontmatter YAML** (`name`, `description`): es lo *unico* que el agente ve por defecto. Por
  eso la `description` debe decir con precision **cuando** usar la skill.
- **Cuerpo Markdown**: las instrucciones completas, que entran al contexto solo al cargarla.

In [ ]:
skills_src = SKILL_REPO / '.claude' / 'skills' / SKILL_SLUG
skills_src.mkdir(parents=True, exist_ok=True)
skill_path = skills_src / 'SKILL.md'

skill_v1 = """---
name: reportero-financiero-cl
description: Genera reportes financieros ejecutivos en espanol chileno con formato estandar. Usala cuando el usuario pida un reporte, resumen o analisis de cifras financieras.
---

# Reportero financiero (Chile)

Eres un analista financiero. Cuando generes un reporte, sigue SIEMPRE estas reglas:

## Estructura obligatoria del reporte
1. **Resumen**: 2-3 frases con la conclusion principal.
2. **Cifras clave**: lista con los montos relevantes (ventas, costos, resultado).
3. **Riesgos**: 2-3 riesgos o alertas.

Cierra con una **recomendacion** accionable en una frase.
"""

skill_path.write_text(skill_v1, encoding='utf-8')
print(f'Escrita {skill_path} ({len(skill_v1)} caracteres)')

### 4. Sube y versiona la skill

`playbook add <ruta>` la deja en *staging* (la ruta debe estar dentro de `.claude/skills/`) y
`playbook submit` la publica. Packmind **versiona automaticamente**: crea una version nueva solo
si el contenido cambio respecto de la ultima (subir contenido identico no duplica la version).

In [ ]:
skill_rel = skill_path.parent.relative_to(SKILL_REPO)   # ruta relativa dentro de .claude/skills/
packmind('playbook', 'add', str(skill_rel), cwd=SKILL_REPO, check=False, retries=3)
packmind('playbook', 'submit', '--no-review', '-m', 'v1: reporte financiero base', cwd=SKILL_REPO, check=False, retries=3)

### 5. Los otros dos artefactos: standard y command

Ahora completamos el trio. Se crean igual (un archivo en su directorio reconocido) y se versionan
igual (`playbook add` + `submit`), pero **se usan distinto**:

- **Standard** (`.claude/rules/<slug>.md`): una **regla siempre activa** (p. ej. la convencion de
  formato de montos). El agente la tiene presente en TODO momento, sin tener que "cargarla".
- **Command** (`.claude/commands/<slug>.md`): una **accion que invoca el usuario** (en Claude Code
  seria `/reporte-trimestral`). Encapsula una tarea recurrente.

> El nombre visible y el *slug* se derivan del encabezado `#` del archivo; mantenlo alineado con
> el nombre del archivo (kebab-case) para que el slug sea predecible.

In [ ]:
# STANDARD: regla SIEMPRE activa -> .claude/rules/<slug>.md
rules_src = SKILL_REPO / '.claude' / 'rules'
rules_src.mkdir(parents=True, exist_ok=True)
standard_path = rules_src / f'{STANDARD_SLUG}.md'
standard_path.write_text("""---
name: formato-montos-clp
description: Convencion de formato para montos en pesos chilenos.
---

# Formato montos CLP

- Expresa los montos en pesos chilenos con separador de miles con punto: `$1.234.567`.
- Antepon siempre el signo `$`. No agregues decimales salvo que la cifra original los tenga.
""", encoding='utf-8')

# COMMAND: accion invocada por el usuario -> .claude/commands/<slug>.md
commands_src = SKILL_REPO / '.claude' / 'commands'
commands_src.mkdir(parents=True, exist_ok=True)
command_path = commands_src / f'{COMMAND_SLUG}.md'
command_path.write_text("""# Reporte trimestral

Genera el reporte financiero del ultimo trimestre para las cifras que entregue el usuario.
Usa la skill `reportero-financiero-cl` y respeta el estandar de formato de montos.
""", encoding='utf-8')

# Versionamos ambos (la skill ya fue subida).
for rel in (standard_path.relative_to(SKILL_REPO), command_path.relative_to(SKILL_REPO)):
    packmind('playbook', 'add', str(rel), cwd=SKILL_REPO, check=False, retries=3)
packmind('playbook', 'submit', '--no-review', '-m', 'Agrega standard de formato y command de reporte', cwd=SKILL_REPO, check=False, retries=3)

### 6. Agrupa los tres artefactos en un *package*

Un *package* reune skills, standards y commands para desplegarlos juntos a otros proyectos.
Creamos uno y agregamos los tres. Packmind asigna el slug del package con *namespace* del espacio
(p. ej. `@global/skills-clase-36`); lo **leemos** de la salida en vez de adivinarlo.

In [ ]:
import re

PACKAGE_NAME = 'Skills Clase 3.6'
ANSI = re.compile(r'\x1b\[[0-9;]*m')   # codigos de color que la CLI mezcla en la salida

create_out = packmind('packages', 'create', PACKAGE_NAME, '-d', 'Artefactos de la leccion 3',
                      cwd=SKILL_REPO, check=False)

# "Skills Clase 3.6" -> @global/skills-clase-36. No lo adivinamos: lo LEEMOS de la salida.
def find_package_slug(*texts):
    for t in texts:
        m = re.search(r'@[\w.-]+/[\w.-]*clase[\w.-]*', ANSI.sub('', t or ''))
        if m:
            return m.group(0)
    return None

packages_out = packmind('packages', 'list', cwd=SKILL_REPO, check=False).stdout
PACKAGE_SLUG = find_package_slug(create_out.stdout, create_out.stderr, packages_out)
if not PACKAGE_SLUG:
    PACKAGE_SLUG = '@global/skills-clase-36'   # ajusta segun el bloque "Install:" de arriba
    print('AVISO: no pude extraer el slug automaticamente; usando', PACKAGE_SLUG)
print('PACKAGE_SLUG =', PACKAGE_SLUG)

# Agregar los tres artefactos al package (idempotente; check=False + reintentos).
packmind('packages', 'add', '--to', PACKAGE_SLUG, '--skill', SKILL_SLUG, cwd=SKILL_REPO, check=False, retries=2)
packmind('packages', 'add', '--to', PACKAGE_SLUG, '--standard', STANDARD_SLUG, cwd=SKILL_REPO, check=False, retries=2)
packmind('packages', 'add', '--to', PACKAGE_SLUG, '--command', COMMAND_SLUG, cwd=SKILL_REPO, check=False, retries=2)

### 7. Despliega el package en el workspace del agente

Simulamos un proyecto **consumidor** distinto (`agent_workdir/`). Tambien necesita `init`; luego
`install` descarga el package y escribe `packmind-lock.json` (el *lockfile* que fija la version
desplegada: el equivalente de `uv.lock` para artefactos). Los tres quedan en `.claude/skills/`,
`.claude/rules/` y `.claude/commands/`.

In [ ]:
import shutil as _shutil

# Consumidor limpio en cada corrida (mismo motivo que en el repo de autoria).
if WORKDIR.exists():
    _shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True, exist_ok=True)
packmind('init', cwd=WORKDIR, stdin_text='2\n', check=False, retries=2)
packmind('install', PACKAGE_SLUG, cwd=WORKDIR, check=False, retries=2)

skills_root = WORKDIR / '.claude' / 'skills'
rules_root = WORKDIR / '.claude' / 'rules'
commands_root = WORKDIR / '.claude' / 'commands'
deployed = skills_root / SKILL_SLUG / 'SKILL.md'


def _ensure(dst: Path, src: Path):
    """Respaldo: garantiza el artefacto en el consumidor aunque `install` no lo deje."""
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        _shutil.copy(src, dst)


_ensure(deployed, skill_path)
_ensure(rules_root / standard_path.name, standard_path)
_ensure(commands_root / command_path.name, command_path)

print('Artefactos desplegados en el consumidor:')
for label, root in (('skills', skills_root), ('rules/standards', rules_root), ('commands', commands_root)):
    files = sorted(root.rglob('*.md')) if root.exists() else []
    print(f'  {label}:', [str(p.relative_to(WORKDIR)) for p in files])

lock = WORKDIR / 'packmind-lock.json'
print('\npackmind-lock.json:')
print(lock.read_text(encoding='utf-8') if lock.exists() else '(sin lockfile; se uso el respaldo local)')

## Como consume el agente cada artefacto

Aqui esta el corazon del *context engineering*, y el contraste entre los tres artefactos:

- El **standard** se inyecta **siempre** en el system prompt (regla permanente).
- La **skill** NO va completa en el prompt: el agente solo ve su nombre+descripcion y carga el
  cuerpo **bajo demanda** con la herramienta `load_skill` (*progressive disclosure*).
- El **command** es lo que el **usuario** dispara para pedir la tarea.

Primero leemos, del workspace desplegado, el catalogo de skills y el texto de los standards.

In [ ]:
import yaml


def discover_skills(skills_root: Path) -> dict:
    """Lee el frontmatter (name/description) de cada SKILL.md desplegado."""
    catalog = {}
    for skill_md in sorted(skills_root.rglob('SKILL.md')):
        text = skill_md.read_text(encoding='utf-8')
        if text.startswith('---'):
            _, front_matter, _body = text.split('---', 2)
            meta = yaml.safe_load(front_matter) or {}
        else:
            meta = {}
        name = meta.get('name', skill_md.parent.name)
        catalog[name] = {'description': meta.get('description', ''), 'path': str(skill_md)}
    return catalog


def load_standards(rules_root: Path) -> str:
    """Concatena los standards desplegados (van SIEMPRE en el contexto)."""
    if not rules_root.exists():
        return ''
    return '\n\n'.join(p.read_text(encoding='utf-8') for p in sorted(rules_root.glob('*.md')))


SKILLS = discover_skills(skills_root)
STANDARDS_TEXT = load_standards(rules_root)
print('Catalogo de skills (lo unico que vera el agente por defecto):')
for name, info in SKILLS.items():
    print(f"- {name}: {info['description']}")
print('\nStandards SIEMPRE activos:', [p.name for p in sorted(rules_root.glob("*.md"))] if rules_root.exists() else [])

### Construimos el agente LangChain

Definimos la tool `load_skill` y un `create_agent` cuyo system prompt lleva los **standards
siempre activos** + el **catalogo de skills** (solo nombre/descripcion). Usamos `gpt-4o-mini`
con `temperature=0` a proposito: honra la temperatura (los modelos gpt-5 solo aceptan el valor
por defecto) y asi la comparacion v1 vs v2 tiene la menor varianza posible.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI


@tool
def load_skill(skill_name: str) -> str:
    """Carga el contenido completo (SKILL.md) de una skill disponible, dado su nombre exacto."""
    info = SKILLS.get(skill_name)
    if info is None:
        return f"No existe la skill '{skill_name}'. Disponibles: {', '.join(SKILLS)}"
    return Path(info['path']).read_text(encoding='utf-8')


def build_agent(catalog: dict, standards_text: str = ''):
    catalogo = '\n'.join(f"- {name}: {info['description']}" for name, info in catalog.items())
    reglas = f'Reglas SIEMPRE activas (standards):\n{standards_text}\n\n' if standards_text.strip() else ''
    system_prompt = (
        'Eres un asistente que puede especializarse cargando skills.\n\n'
        f'{reglas}'
        'Skills disponibles (nombre: cuando usarla):\n'
        f'{catalogo}\n\n'
        'Antes de resolver una tarea cubierta por una skill, cargala con la herramienta '
        '`load_skill` (usando su nombre exacto) y sigue sus instrucciones al pie de la letra. '
        'Respeta SIEMPRE los standards de arriba.'
    )
    return create_agent(
        model=ChatOpenAI(model='gpt-4o-mini', temperature=0),
        tools=[load_skill],
        system_prompt=system_prompt,
    )


agent = build_agent(SKILLS, STANDARDS_TEXT)
print('Agente construido |', len(SKILLS), 'skills |', 'standards activos:', bool(STANDARDS_TEXT.strip()))

## Ejecutamos: el *command* dispara al agente

"Invocar el command" = usar su contenido como instruccion del usuario. En una corrida veras el
trio en accion: el **command** pide la tarea, el agente carga la **skill** con `load_skill`
(*progressive disclosure*) y respeta el **standard** de formato que ya venia en el system prompt.

In [ ]:
def command_message(workdir: Path) -> str:
    cmd = (workdir / '.claude' / 'commands' / f'{COMMAND_SLUG}.md').read_text(encoding='utf-8')
    return cmd + '\n\nCifras del trimestre: ventas 45000000 CLP, costos 31000000 CLP.'


def run_agent(agent, task: str, etiqueta: str = '') -> str:
    result = agent.invoke({'messages': [{'role': 'user', 'content': task}]})
    messages = result['messages']
    for msg in messages:
        tool_calls = getattr(msg, 'tool_calls', None)
        if tool_calls:
            for tc in tool_calls:
                print(f"[tool_call] {tc['name']}({tc['args']})")
        elif msg.__class__.__name__ == 'ToolMessage':
            print(f"[tool_result] {msg.name}: {len(str(msg.content))} caracteres cargados al contexto")
    final = messages[-1].content
    print(f'\n=== Respuesta del agente {etiqueta} ===\n')
    print(final)
    return final


TASK = command_message(WORKDIR)
respuesta_v1 = run_agent(agent, TASK, etiqueta='(skill v1)')

> **Revisa la traza en LangSmith** (proyecto `clase-3-6-l3-skills`): veras el paso en que el
> agente llama a `load_skill` antes de redactar. Ese es el enlace con las lecciones 1 y 2: ahora
> puedes *evaluar* y *auditar* no solo el prompt, sino tambien **que version de que skill** se
> uso en cada corrida.

## Versionamos: skill v2

Cambiamos el comportamiento de la skill (misma `name`/`description`, distinto cuerpo): ahora el
resumen debe ser 3 bullets, se exige el **margen bruto en %** y se agrega un descargo de
responsabilidad. Al resubir, Packmind crea **una nueva version** automaticamente. (Los standards
y commands quedan igual: versionar uno no toca a los demas.)

In [ ]:
skill_v2 = """---
name: reportero-financiero-cl
description: Genera reportes financieros ejecutivos en espanol chileno con formato estandar. Usala cuando el usuario pida un reporte, resumen o analisis de cifras financieras.
---

# Reportero financiero (Chile) -- v2

Eres un analista financiero. Cuando generes un reporte, sigue SIEMPRE estas reglas:

## Estructura obligatoria del reporte
1. **Resumen ejecutivo**: exactamente 3 bullets con lo esencial.
2. **Cifras clave**: ventas, costos, resultado y **margen bruto en %** (resultado / ventas).
3. **Riesgos**: 2-3 riesgos o alertas.

Cierra con una **recomendacion** accionable y este descargo textual:
> Este reporte es informativo y no constituye asesoria financiera.
"""

skill_path.write_text(skill_v2, encoding='utf-8')
skill_rel = skill_path.parent.relative_to(SKILL_REPO)
packmind('playbook', 'add', str(skill_rel), cwd=SKILL_REPO, check=False, retries=3)
packmind('playbook', 'submit', '--no-review', '-m', 'v2: resumen ejecutivo + margen bruto + descargo', cwd=SKILL_REPO, check=False, retries=3)

### Redespliega y observa el lockfile

Volvemos a instalar el package en el consumidor: el `packmind-lock.json` deberia apuntar ahora a
la nueva version de la skill. Comparamos el lockfile antes/despues y recargamos el catalogo.

In [ ]:
lock_v1 = lock.read_text(encoding='utf-8') if lock.exists() else '(sin lockfile)'

packmind('install', PACKAGE_SLUG, cwd=WORKDIR, check=False, retries=2)

# Respaldo: garantizamos que la skill desplegada refleje la v2.
if not deployed.exists() or 'v2' not in deployed.read_text(encoding='utf-8'):
    deployed.parent.mkdir(parents=True, exist_ok=True)
    deployed.write_text(skill_path.read_text(encoding='utf-8'), encoding='utf-8')

lock_v2 = lock.read_text(encoding='utf-8') if lock.exists() else '(sin lockfile)'
print('--- packmind-lock.json (v1) ---\n', lock_v1)
print('\n--- packmind-lock.json (v2) ---\n', lock_v2)

# Recargar catalogo y standards (el cuerpo de la skill cambio; el standard sigue igual).
SKILLS = discover_skills(skills_root)
STANDARDS_TEXT = load_standards(rules_root)

## Reejecutamos el mismo *command* con la skill v2

Reconstruimos el agente (catalogo + standards se regeneran desde lo desplegado) y disparamos el
**mismo** command. Lo unico que cambio es la version de la skill.

In [ ]:
agent = build_agent(SKILLS, STANDARDS_TEXT)
respuesta_v2 = run_agent(agent, TASK, etiqueta='(skill v2)')

## Comparacion y cierre

Compara `respuesta_v1` y `respuesta_v2`: con el mismo command y `temperature=0`, la v2 debe traer
el resumen en 3 bullets, el margen bruto en % y el descargo. El cambio viene **solo** de versionar
la skill; el command, el standard y el codigo del agente no se tocaron. En **app.packmind.ai**
(seccion Skills) veras el historial: v1 y v2 con sus timestamps y mensajes.

### El hilo de la clase
- **Leccion 2:** versionaste *prompts* (Prompt Hub).
- **Leccion 3:** versionaste los tres artefactos de un agente con Packmind:
  - **skill** (agent-discovered, bajo demanda),
  - **standard** (siempre activo en el contexto),
  - **command** (invocado por el usuario).
- **Lockfile (`packmind-lock.json`):** reproducibilidad -- fijas *que version* corre en produccion.
- **Progressive disclosure:** context engineering -- el contexto recibe solo lo relevante, bajo demanda.
- **Trazas (LangSmith):** evidencia de *cuando* y *que version* de skill se uso (enlaza con evaluacion).